<a href="https://colab.research.google.com/github/NguyenTNgan/spatial-temporal-defect-segmentation/blob/main/notebooks/thermal_dataset_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
ZIP_PATH="/content/drive/MyDrive/Suzlon/Thermal Inspection Dataset for Defect Segmentation in CFRP Laminates.zip"

BASE="/content/dataset"
ORIGINAL="/content/original"
ANNOTATED="/content/annotated"


!rm -rf /content/dataset /content/original /content/annotated
!mkdir -p /content/dataset

!unzip -q "$ZIP_PATH" -d /content/dataset

import glob

orig_zip = glob.glob(BASE + "/**/originalData.zip", recursive=True)[0]
ann_zip  = glob.glob(BASE + "/**/annotatedData.zip", recursive=True)[0]

print("Original ZIP:", orig_zip)
print("Annotated ZIP:", ann_zip)


!mkdir -p /content/original /content/annotated

!unzip -q "$orig_zip" -d /content/original
!unzip -q "$ann_zip" -d /content/annotated

Original ZIP: /content/dataset/Thermal Inspection Dataset for Defect Segmentation in CFRP Laminates/originalData.zip
Annotated ZIP: /content/dataset/Thermal Inspection Dataset for Defect Segmentation in CFRP Laminates/annotatedData.zip


In [15]:
import os
import shutil
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

In [16]:
# Clean old flatten folders

shutil.rmtree('/content/original_flat', ignore_errors=True)
shutil.rmtree('/content/annotated_flat', ignore_errors=True)

print('Old flat folders deleted.')

# Remove MacOS junk files
def flatten(src, dst):

    os.makedirs(dst, exist_ok=True)

    copied = 0

    for root, _, files in os.walk(src):

        if '__MACOSX' in root:
            continue

        for f in files:

            if f.startswith('._'):
                continue

            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):

                src_path = os.path.join(root, f)
                dst_path = os.path.join(dst, f)

                shutil.copy(src_path, dst_path)

                copied += 1

    print(f'Copied {copied} files from {src}')

# Flatten datasets
flatten('/content/original', '/content/original_flat')
flatten('/content/annotated', '/content/annotated_flat')

# File counts
original_files = os.listdir('/content/original_flat')
annotated_files = os.listdir('/content/annotated_flat')

print('\nDataset counts')
print('Original flat:', len(original_files))
print('Annotated flat:', len(annotated_files))

print('\nSample original:', original_files[:5])
print('Sample annotated:', annotated_files[:5])

# Match images & Masks
img_dir = '/content/original_flat'
mask_dir = '/content/annotated_flat'

img_files = set(os.listdir(img_dir))
mask_files = set(os.listdir(mask_dir))

common_files = sorted(list(img_files & mask_files))

unmatched_images = sorted(list(img_files - mask_files))
unmatched_masks = sorted(list(mask_files - img_files))

print('\nMatching results')
print('Matched pairs:', len(common_files))
print('Unmatched images:', len(unmatched_images))
print('Unmatched masks:', len(unmatched_masks))

# Create file paths
image_paths = [
    os.path.join(img_dir, f)
    for f in common_files
]

mask_paths = [
    os.path.join(mask_dir, f)
    for f in common_files
]

# Dataset
class ThermalDataset(Dataset):

    def __init__(self, image_paths, mask_paths):

        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Read image & mask
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, 0)

        if img is None:
            raise ValueError(f'Failed to load image: {img_path}')

        if mask is None:
            raise ValueError(f'Failed to load mask: {mask_path}')

        # Convert BGR -> RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Normalize
        img = img / 255.0
        mask = mask / 255.0

        img = torch.tensor(
            img,
            dtype=torch.float32
        ).permute(2, 0, 1)

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        ).unsqueeze(0)

        return img, mask


# Create dataset
dataset = ThermalDataset(
    image_paths=image_paths,
    mask_paths=mask_paths
)

print('\nFinal dataset')
print('Dataset size:', len(dataset))


# Dataloader
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)


for imgs, masks in loader:

    print('Images shape:', imgs.shape)
    print('Masks shape:', masks.shape)

    break

Old flat folders deleted.
Copied 1034 files from /content/original
Copied 1034 files from /content/annotated

Dataset counts
Original flat: 1034
Annotated flat: 1034

Sample original: ['714.png', '100.png', '1043.png', '580.png', '654.png']
Sample annotated: ['714.png', '100.png', '1043.png', '580.png', '654.png']

Matching results
Matched pairs: 1034
Unmatched images: 0
Unmatched masks: 0

Final dataset
Dataset size: 1034
Images shape: torch.Size([8, 3, 234, 234])
Masks shape: torch.Size([8, 1, 234, 234])
